# 09 - Bakery manual review

This notebook reviews the bakery verification results from Notebook 08.

The purpose is to identify specific classifications that may require manual checking and to maintain a single manual-review record.

In [ ]:
from pathlib import Path
from shutil import copy2

import pandas as pd

# Main verification folder
VERIFICATION_FOLDER = Path("../data/business/interim/ai_verification_v2")

# Notebook 08 output and Notebook 09 output
AI_RESULTS_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_results.csv")
MANUAL_REVIEW_FOLDER = (VERIFICATION_FOLDER / "manual_review")
MANUAL_REVIEW_FOLDER.mkdir(parents=True, exist_ok=True)

BACKUP_FOLDER = (MANUAL_REVIEW_FOLDER / "backups")
BACKUP_FOLDER.mkdir(parents=True, exist_ok=True)

MANUAL_REVIEW_PATH = (MANUAL_REVIEW_FOLDER / "bakery_manual_review.xlsx")


EXPECTED_TOTAL = 28847

## Load AI verification results

The AI verification results are loaded as the source dataset for manual-review diagnostics.

In [4]:
ai_results = pd.read_csv(AI_RESULTS_PATH)

ai_results = (ai_results
              .sort_values("BakeryRank")
              .reset_index(drop=True))

print(f"AI verification rows loaded: {len(ai_results)}")

print(f"Highest BakeryRank available: {ai_results['BakeryRank'].max()}")

print(f"Verification completion: {len(ai_results) / EXPECTED_TOTAL:.2%}")

AI verification rows loaded: 18452
Highest BakeryRank available: 18507
Verification completion: 63.97%


## Check loaded verification results

A small number of checks are used to make sure the verification results are suitable for manual review.

In [11]:
ESSENTIAL_COLUMNS = [
    "BusinessName",
    "FHRSIDRep",
    "BusinessType",
    "Address",
    "PostCode",
    "LocalAuthorityName",
    "BakeryRank",
    "BakeryScore",
    "StoreCount",
    "LocationMatch",
    "AIStatus",
    "PhysicalRetail",
    "BakeryFocus",
    "AIVerdict",
    "AIReason"]

missing_columns = [column
                   for column in ESSENTIAL_COLUMNS
                   if column not in ai_results.columns]

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

if ai_results["BakeryRank"].isna().any():
    raise ValueError("Missing BakeryRank values found.")

if ai_results["BakeryRank"].duplicated().any():
    raise ValueError("Duplicate BakeryRank values found.")

print("Verification results ready for review.")

Verification results ready for review.


## Inspect AI verification results

The overall verification outputs are inspected before any manual-review rules are defined.

This provides an overview of the AI classifications and the individual verification fields used to produce them.

In [7]:
VERIFICATION_COLUMNS = [
    "AIVerdict",
    "LocationMatch",
    "AIStatus",
    "PhysicalRetail",
    "BakeryFocus"]


for column in VERIFICATION_COLUMNS:

    summary = (ai_results[column]
               .value_counts(dropna=False)
               .rename("Count")
               .to_frame())

    summary["Percent"] = (summary["Count"] / len(ai_results) * 100).round(1)

    print(f"\n{column}")

    display(summary)


AIVerdict


,Count,Percent
AIVerdict,,
NOT_BAKERY,11886,64.4
UNCLEAR,4997,27.1
BAKERY,1569,8.5



LocationMatch


,Count,Percent
LocationMatch,,
YES,13895,75.3
UNCLEAR,4557,24.7



AIStatus


,Count,Percent
AIStatus,,
ACTIVE,13350,72.3
UNCLEAR,4569,24.8
INACTIVE,533,2.9



PhysicalRetail


,Count,Percent
PhysicalRetail,,
NO,6807,36.9
YES,6254,33.9
UNCLEAR,5391,29.2



BakeryFocus


,Count,Percent
BakeryFocus,,
NO,10288,55.8
UNCLEAR,6312,34.2
YES,1852,10.0


### Relationship between verdicts and verification fields

The final AI verdict is compared with each of the individual verification fields.

In [10]:
VERDICT_FIELDS = [
    "LocationMatch",
    "AIStatus",
    "PhysicalRetail",
    "BakeryFocus"]

for column in VERDICT_FIELDS:
    print(f"\nAIVerdict by {column}")

    comparison = pd.crosstab(ai_results["AIVerdict"], ai_results[column], margins=True)

    display(comparison)


AIVerdict by LocationMatch


LocationMatch,UNCLEAR,YES,All
AIVerdict,,,
BAKERY,0,1569,1569
NOT_BAKERY,0,11886,11886
UNCLEAR,4557,440,4997
All,4557,13895,18452



AIVerdict by AIStatus


AIStatus,ACTIVE,INACTIVE,UNCLEAR,All
AIVerdict,,,,
BAKERY,1569,0,0,1569
NOT_BAKERY,11340,533,13,11886
UNCLEAR,441,0,4556,4997
All,13350,533,4569,18452



AIVerdict by PhysicalRetail


PhysicalRetail,NO,UNCLEAR,YES,All
AIVerdict,,,,
BAKERY,0,0,1569,1569
NOT_BAKERY,6778,542,4566,11886
UNCLEAR,29,4849,119,4997
All,6807,5391,6254,18452



AIVerdict by BakeryFocus


BakeryFocus,NO,UNCLEAR,YES,All
AIVerdict,,,,
BAKERY,0,0,1569,1569
NOT_BAKERY,10288,1358,240,11886
UNCLEAR,0,4954,43,4997
All,10288,6312,1852,18452


### Business types by AI verdict

The FHRS business types are compared across the AI verdicts to understand the
types of establishments appearing in each classification.

In [12]:
business_type_summary = pd.crosstab(ai_results["BusinessType"], ai_results["AIVerdict"], margins=True)

business_type_summary = (business_type_summary
                         .sort_values("BAKERY", ascending=False))

display(business_type_summary)

AIVerdict,BAKERY,NOT_BAKERY,UNCLEAR,All
BusinessType,,,,
All,1569,11886,4997,18452
Restaurant/Cafe/Canteen,708,3149,766,4623
Retailers - other,433,2417,1129,3979
Takeaway/sandwich shop,206,677,289,1172
Other catering premises,95,1619,1846,3560
Manufacturers/packers,89,353,131,573
Mobile caterer,26,315,429,770
Retailers - supermarkets/hypermarkets,4,152,15,171
Hotel/bed & breakfast/guest house,3,304,18,325


### Inspect example classifications

Examples from each AI verdict are inspected to understand how the verification fields and evidence reasons are being used in individual cases.

In [ ]:
SAMPLE_COLUMNS = [
    "BakeryRank",
    "BakeryScore",
    "BusinessName",
    "BusinessType",
    "Address",
    "PostCode",
    "LocationMatch",
    "AIStatus",
    "PhysicalRetail",
    "BakeryFocus",
    "AIVerdict",
    "AIReason"]

for verdict in ["BAKERY", "NOT_BAKERY", "UNCLEAR"]:
    sample = (ai_results[ai_results["AIVerdict"] == verdict]
        .sample(n=10, random_state=42)
        .sort_values("BakeryRank"))

    print(f"\n{verdict}")
    display(sample[SAMPLE_COLUMNS])


BAKERY


,BakeryRank,BakeryScore,BusinessName,BusinessType,Address,PostCode,LocationMatch,AIStatus,PhysicalRetail,BakeryFocus,AIVerdict,AIReason
1844,1845,0.500999,El gran sazon latino bakery,Takeaway/sandwich shop,"80-82, Walworth Road, London",SE1 6SW,YES,ACTIVE,YES,YES,BAKERY,News articles and menu listings confirm a Colo...
2673,2674,0.286518,Wonderful Patisserie,Restaurant/Cafe/Canteen,"45 GERRARD STREET, LONDON",W1D 5QQ,YES,ACTIVE,YES,YES,BAKERY,Wonderful Patisserie is a traditional Chinese ...
6719,6720,0.065706,Dough Aldgate East,Restaurant/Cafe/Canteen,"Dough, 8 Piazza Walk, London",E1 8FU,YES,ACTIVE,YES,YES,BAKERY,Dough Aldgate East is a Chinese dough house sp...
7079,7080,0.061423,Sponge Kitchens,Retailers - other,"106 Wickham Road, Beckenham",BR3 6QH,YES,ACTIVE,YES,YES,BAKERY,Sponge Kitchens is a family-run bakery with mu...
7284,7285,0.059270,Dum Dum Donutterie,Retailers - other,"Euston Railway Station, Eversholt Street, London",NW1 2HS,YES,ACTIVE,YES,YES,BAKERY,Dum Dum Donutterie at Euston Station is a phys...
7855,7856,0.053989,Grains & Greens,Other catering premises,NaN,NW11,YES,ACTIVE,YES,YES,BAKERY,Grains & Greens is a wholefoods store and cafe...
9019,9020,0.045469,Cream Lab,Restaurant/Cafe/Canteen,"DUMBARTON HOUSE 68 OXFORD STREET, LONDON",W1D 1BN,YES,ACTIVE,YES,YES,BAKERY,"Cream Lab Cafe offers pastries, doughnuts, cak..."
10140,10141,0.039705,Twisted Sister,Restaurant/Cafe/Canteen,"69 St Johns Road, London, Wandsworth",SW11 1QX,YES,ACTIVE,YES,YES,BAKERY,Twisted Sister is an independent community caf...
15252,15254,0.026622,Bagel with Shake/Tazah Simply Fresh/Wraps & Ro...,Takeaway/sandwich shop,"183 Church Road, London",NW10 9EE,YES,ACTIVE,YES,YES,BAKERY,Bagel with Shake is a takeaway/sandwich shop t...
18016,18060,0.022814,RR TASTY,Restaurant/Cafe/Canteen,"88 Victoria Road, Surbiton",KT6 4NS,YES,ACTIVE,YES,YES,BAKERY,Official website shows a physical shop selling...



NOT_BAKERY


,BakeryRank,BakeryScore,BusinessName,BusinessType,Address,PostCode,LocationMatch,AIStatus,PhysicalRetail,BakeryFocus,AIVerdict,AIReason
1132,1133,0.708291,Emma Dodi Cakes,Other catering premises,NaN,SW4,YES,ACTIVE,NO,NO,NOT_BAKERY,"Emma Dodi Cakes operates from a London studio,..."
5022,5023,0.098159,La Latteria- Samia Dairy Limited,Manufacturers/packers,"5 & 7 Cumberland Business Park, Cumberland Ave...",NW10 7RT,YES,ACTIVE,NO,NO,NOT_BAKERY,La Latteria is a dairy that manufactures and d...
6434,6435,0.069544,Zambrero Twickenham,Restaurant/Cafe/Canteen,"30 London Road, Twickenham, Richmond Upon Thames",TW1 3RR,YES,ACTIVE,YES,NO,NOT_BAKERY,Zambrero Twickenham is a Mexican restaurant pr...
7015,7016,0.062039,The brotherhood games ltd,Restaurant/Cafe/Canteen,"210-212, Southwark Park Road, London",SE16 3RX,YES,ACTIVE,YES,NO,NOT_BAKERY,The Brotherhood Games is a hobby store and caf...
8966,8967,0.045746,Glass Door,Other catering premises,"Church Hall, 117 Queen's Gate, LONDON",SW7 5LP,YES,ACTIVE,NO,NO,NOT_BAKERY,Glass Door is a homeless charity providing sup...
9926,9927,0.040720,Shelly Tots Pre-School,Caring Premises,"Mountfield Community Centre 17 Sandway Road, O...",BR5 3TU,YES,ACTIVE,NO,NO,NOT_BAKERY,This is a pre-school located in a community ce...
11539,11540,0.034663,Sujathaa Impex Ltd,Importers/Exporters,"Unit 7 The Acorn Centre, Roebuck Road, Hainault",IG6 3TU,YES,ACTIVE,NO,UNCLEAR,NOT_BAKERY,Sujathaa Impex Ltd is an importer/exporter wit...
11787,11788,0.033943,Memo Cash and Carry Ltd,Distributors/Transporters,"3 Argall Avenue, Leyton, London",E10 7QE,YES,ACTIVE,NO,UNCLEAR,NOT_BAKERY,Memo Cash and Carry Ltd is a wholesale distrib...
13378,13379,0.029976,Pepe nero,Restaurant/Cafe/Canteen,"565 Chiswick High Road, Chiswick",W4 3AY,YES,ACTIVE,YES,NO,NOT_BAKERY,"Pepe Nero is a pizzeria offering Italian food,..."
18230,18275,0.022631,Balls Brothers Austin Friars,Pub/bar/nightclub,"10-11 Austin Friars, London",EC2N 2HG,YES,ACTIVE,YES,NO,NOT_BAKERY,"This is a Balls Brothers bar and restaurant, n..."



UNCLEAR


,BakeryRank,BakeryScore,BusinessName,BusinessType,Address,PostCode,LocationMatch,AIStatus,PhysicalRetail,BakeryFocus,AIVerdict,AIReason
242,243,0.954953,Maison De Bakery,Other catering premises,NaN,CR0,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"The generic name ""Maison De Bakery"" with a par..."
498,499,0.882226,Meein Cake,Other catering premises,Food Exchange New Covent Garden Market Nine El...,SW8 5EL,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"The business name is generic, and the address ..."
1843,1844,0.501676,The Shortbread Studio,Restaurant/Cafe/Canteen,NaN,RM13,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,No reliable evidence found to identify an acti...
2093,2094,0.430538,Sweet Mum,Other catering premises,NaN,SE28,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"No reliable information found for ""Sweet Mum"" ..."
2420,2421,0.349061,Caroline's Cupcakes,Retailers - other,NaN,DA16,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"Generic name ""Caroline's Cupcakes"" with only a..."
4033,4034,0.135224,Croissant,Restaurant/Cafe/Canteen,"2 Chichele Road, London",NW2 3DA,YES,ACTIVE,UNCLEAR,UNCLEAR,UNCLEAR,"""Croissant"" is a generic name, and while the a..."
8861,8862,0.046411,Bubble Bonbon,Restaurant/Cafe/Canteen,NaN,BR6,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"The generic name ""Bubble Bonbon"" with only a p..."
12298,12299,0.032576,Wafflehub,Mobile caterer,NaN,SE28,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"""Wafflehub"" with a partial postcode and ""Mobil..."
15595,15597,0.026091,Prince Of Egypt Catering,Other catering premises,NaN,W3 6,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"""Prince Of Egypt Catering"" with a partial post..."
17885,17929,0.022944,Peter Beagan,Mobile caterer,NaN,TW7,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,UNCLEAR,"The name ""Peter Beagan"" is too generic, and wi..."
